# Welcome!
This notebook demonstrates how to develop a conversational system that uses a deep knowledge base about hotels, combining structured instance-level data and an ontological model. The knowledge graph (KG) and ontology enable reasoning to enhance dialogue response generation. The task involves integrating a GraphRAG-like approach to query the knowledge base and generate accurate, context-aware responses.

Specifically, the notebook has the following steps:

1. **Setup**: Loading the knowledge graph, dialogues, and required libraries (e.g., OWLAPY).
2. **Analyzing the knowledge graph**: Exploring its structure and entities using OWLAPY queries.
3. **Extending the ontology**: Adding TBox information for expressive reasoning.
4. **Creating dialogues**: Create dialogues based on the examples. Write 5 simple dialogues and 5 more detailed ones to showcase different types of interactions.
5. **Combining ontology and KG data**: Deploying an OWL reasoner to perform class-expression queries.
6. **Query generation with LLMs**: Using an LLM (e.g., Llama3.2) to generate or assist in creating queries against the KG.
7. **Generating responses**: Summarizing retrieved data into dialogue responses using a KG-augmented RAG approach.
8. **Evaluation**: Assessing the system's performance using metrics like intersection-over-union scores.

## Assignment
The goal of this assignment is to develop a logic-enhanced conversational system that retrieves and reasons over domain knowledge to assist in dialogue response generation. You will focus on both the technical aspects of KG+ontology reasoning and the integration with LLMs for robust responses.

### Assignment Steps
1. **Analyze the provided knowledge graph and dialogues**:
   - Explore the KG's entities, properties, and relevance to the dialogues.
   - Identify opportunities where ontology reasoning enhances dialogue responses.
2. **Extend the ontology**:
   - Add expressive TBox information to support meaningful inferences.
3. **Deploy the reasoning environment**:
   - Use OWLAPY to combine the KG (as ABox) with the ontology for reasoning-based queries.
4. **Generate class-expression queries**:
   - Use instruction-based, few-shot prompting with Llama3.2 to produce or assist in creating the queries.
5. **Summarize results into dialogue responses**:
   - Apply KG-augmented RAG to generate user-facing answers based on reasoning results.
6. **Evaluate the system**:
   - Use appropriate metrics, including intersection-over-union scores for set-based answers.

## Report
Write a **5-page report** in LNCS format that includes:

1. **Introduction**: Background on conversational systems with LLMs and the role of reasoning over domain knowledge.
2. **Methodology**: A detailed description of your approach, including diagrams and examples.
3. **Results**: Evaluation findings from the implemented steps.
4. **Discussion**: Strengths and weaknesses of your approach, lessons learned, and potential improvements.

Make sure to use the following template: [Springer Lecture Notes in Computer Science](https://www.overleaf.com/latex/templates/springer-lecture-notes-in-computer-science/kzwwpvhwnvfj)


## Grading
Your work will be evaluated based on:

1. **Code Implementation (30%)**: Quality and functionality of the logic-enhanced conversational system.
2. **Report (70%)**: Depth of analysis and clarity in presenting methods, results, and lessons learned.

## Kaggle Environment Notes
To ensure smooth execution:
- Load the required data into `/kaggle/input/`.
- Use `/kaggle/working/` for saving temporary files.
- Turn on GPUs and internet connectivity when necessary, and follow best practices for resource management.

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input director

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

##### For colab: setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Install packages

##### For colab pip installs

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama parsimonious rdflib SPARQLWrapper owlapy

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tgz
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 18.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip install jpype1==1.5.2
!pip install owlapy==1.5.1
!pip install ollama

# Import libraries


In [2]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(10)

!ollama pull llama3.2

In [21]:
import os
import re
import ollama
from parsimonious.exceptions import IncompleteParseError
from rdflib import Graph, Literal, Namespace, URIRef
from SPARQLWrapper import JSON, SPARQLWrapper

from owlapy import dl_to_owl_expression, manchester_to_owl_expression
from owlapy.class_expression import (
    OWLClass,
    OWLObjectHasValue,
    OWLObjectIntersectionOf,
    OWLObjectSomeValuesFrom
)
from owlapy.iri import IRI
from owlapy.owl_axiom import OWLObjectPropertyAssertionAxiom, OWLSubObjectPropertyOfAxiom
from owlapy.owl_individual import OWLNamedIndividual
from owlapy.owl_ontology import Ontology
from owlapy.owl_property import OWLObjectProperty
from owlapy.owl_reasoner import StructuralReasoner, SyncReasoner

working_dir = "/content/drive/MyDrive/psai"
ttl_file_path = os.path.join(working_dir, "extended_data.ttl")
owl_file_path = os.path.join(working_dir, "extended_data.owl")

graph = Graph()
graph.parse(ttl_file_path, format="turtle")

<Graph identifier=N32ecca4c14eb4146991fa2def56e72be (<class 'rdflib.graph.Graph'>)>

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from rdflib import Graph
from owlready2 import World



# 1. Analyze the provided knowledge graph (data.ttl).

In [ ]:
## the provided knowledge graph is in turtle (.ttl) format, which owlready2 has trouble
## parsing correctly in this environment. to avoid this issue, we first load the file
## using rdflib (which fully supports turtle), convert it to n-triples,
## and then load the converted graph into owlready2 for analysis.
## for the record owlready2 is an inner library used by owlapy.

from rdflib import Graph
from owlready2 import World

src = "/kaggle/input/w1-dataset3/extended_data.ttl"
dst = "/kaggle/working/data.nt"   ## .nt is the file format for n-triples

g = Graph()

g.parse(src, format="turtle")   ## here we parse the .ttl
g.serialize(destination=dst, format="nt")   # here its converted into .nt

world = World()
onto = world.get_ontology(f"file://{dst}").load(format="ntriples")

print("loaded into:", onto.base_iri)

In [ ]:
from collections import Counter
from rdflib import URIRef, Literal
import pandas as pd

# helper funcs
def is_uri(x):
    return isinstance(x, URIRef)

def is_lit(x):
    return isinstance(x, Literal)

def shorten(term, graph):
    # compact display using namespaces when possible

    if isinstance(term, URIRef):
        try:
            return term.n3(graph.namespace_manager)
        except Exception:
            return str(term)
    if isinstance(term, Literal):
        if term.language:
            return f"\"{str(term)[:60]}\"@{term.language}"
        if term.datatype:
            return f"\"{str(term)[:60]}\"^^{term.datatype}"
        return f"\"{str(term)[:60]}\""
    return str(term)


triples = list(g.triples((None, None, None)))
print("--- basic info ---")
print("triples:", len(triples))

subjects = set(s for s, p, o in triples)
predicates = set(p for s, p, o in triples)
objects = set(o for s, p, o in triples)

uris = set(x for x in subjects.union(objects) if is_uri(x))
lits = set(x for x in objects if is_lit(x))

print("unique subjects:", len(subjects))
print("unique predicates:", len(predicates))
print("unique objects:", len(objects))
print("unique URI nodes (subjects U objects):", len(uris))
print("unique literal nodes (objects):", len(lits))

pred_counts = Counter(p for s, p, o in triples)
top_preds = pred_counts.most_common(30)

print()
print("--- top predicates (by triple count) ---")

df_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "count"]
)
display(df_preds.head(30))

RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

type_triples = list(g.triples((None, RDF_TYPE, None)))
class_counts = Counter(o for s, p, o in type_triples if is_uri(o))

print()
print("--- types / classes (rdf:type) ---")
print("rdf:type triples:", len(type_triples))
print("distinct classes:", len(class_counts))

df_classes = pd.DataFrame(
    [(str(cls), shorten(cls, g), c) for cls, c in class_counts.most_common()],
    columns=["class_iri", "class", "instances_count"]
)
display(df_classes.head(30))

datatype_counts = Counter()
lang_counts = Counter()
lit_pred_counts = Counter()
lit_lengths = []

for s, p, o in triples:
    if is_lit(o):
        lit_pred_counts[p] += 1
        if o.datatype:
            datatype_counts[o.datatype] += 1
        else:
            datatype_counts[None] += 1
        if o.language:
            lang_counts[o.language] += 1
        lit_lengths.append(len(str(o)))

print()
print("--- literals ---")
print("literal objects:", sum(lit_pred_counts.values()))
print("predicates with literals:", len(lit_pred_counts))
print("avg literal length:", (sum(lit_lengths) / len(lit_lengths)) if lit_lengths else 0)

print("top literal predicates:")
for p, c in lit_pred_counts.most_common(20):
    print(f"{c:>7}  {shorten(p, g)}")

print("top datatypes:")
for dt, c in datatype_counts.most_common(15):
    dt_name = "no-datatype" if dt is None else shorten(dt, g)
    print(f"{c:>7}  {dt_name}")

print("top languages:")
for lang, c in lang_counts.most_common(10):
    print(f"{c:>7}  {lang}")

df_lit_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in lit_pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "literal_count"]
)
display(df_lit_preds.head(30))

--- basic info ---
triples: 9237
unique subjects: 1789
unique predicates: 14
unique objects: 870
unique URI nodes (subjects U objects): 1792
unique literal nodes (objects): 476

--- top predicates (by triple count) ---


,predicate_iri,predicate,count
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,rdf:type,3751
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:hasFacility,1640
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:location,1085
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:userRating,1000
4,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nationality,472
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:restaurantType,305
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:inCity,267
8,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:diet,137
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nextTo,34



--- types / classes (rdf:type) ---
rdf:type triples: 3751
distinct classes: 32


,class_iri,class,instances_count
0,http://www.w3.org/2002/07/owl#NamedIndividual,owl:NamedIndividual,1746
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Restaurant,506
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hotel,344
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Camping_Site,342
4,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hostel,314
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Neighbourhood,267
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Museum,55
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:City,34
8,http://www.w3.org/2002/07/owl#Class,owl:Class,33
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Trainstation,22



--- literals ---
literal objects: 485
predicates with literals: 1
avg literal length: 13.393814432989691
top literal predicates:
    485  rdfs:label
top datatypes:
    485  no-datatype
top languages:


,predicate_iri,predicate,literal_count
0,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485


# 2. Create a small ontology that can support expressive inference about hotels and analyse the dialogues (examples.txt).

In [ ]:
import os
import re

import ollama
from parsimonious.exceptions import IncompleteParseError
from rdflib import Graph, Literal, Namespace, URIRef
from SPARQLWrapper import JSON, SPARQLWrapper

from owlapy import dl_to_owl_expression, manchester_to_owl_expression
from owlapy.class_expression import (
    OWLClass,
    OWLObjectHasValue,
    OWLObjectIntersectionOf,
    OWLObjectSomeValuesFrom
)
from owlapy.iri import IRI
from owlapy.owl_axiom import OWLObjectPropertyAssertionAxiom, OWLSubObjectPropertyOfAxiom
from owlapy.owl_individual import OWLNamedIndividual
from owlapy.owl_ontology import Ontology
from owlapy.owl_property import OWLObjectProperty
from owlapy.owl_reasoner import StructuralReasoner, SyncReasoner

c:\Users\hippo\miniconda3\envs\FLAI\Lib\site-packages\owlapy\static_funcs.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


##### Convert ttl to owl

In [ ]:
file_name = "extended_data" # Change to your filename (without extension)
g = Graph().parse(f"{file_name}.ttl", format="turtle")
g.serialize(destination=f"{file_name}.owl", format="xml")

<Graph identifier=N7a7d6901e81d47189f90109c4fd00f5b (<class 'rdflib.graph.Graph'>)>

##### Helper functions

In [4]:
def get_local_uri(label_text, file_path=owl_file_path):
    g = Graph()
    g.parse(file_path, format="xml")

    q = """
    SELECT ?s WHERE {
        ?s ?p ?label .
        FILTER(STR(?label) = "%s")
    } LIMIT 1
    """ % label_text

    results = g.query(q)
    for row in results:
        return row.s
    return None

uri = get_local_uri("Gulf") # test
print(uri)

http://www.wikidata.org/entity/Q1322134


In [5]:
def resolve_manchester_fillers(query):
    keywords = {'and', 'or', 'some', 'only', 'value', 'min', 'max', 'exactly', 'not', 'that'}

    def replacement_logic(match):
        word = match.group(0)

        if word.lower() in keywords:
            return word

        uri = get_local_uri(word)

        if uri:
            uri_str = str(uri)
            if "wikidata.org" in uri_str:
                return uri_str #uri_str.split('/')[-1]

        return word

    return re.sub(r'\b\w+\b', replacement_logic, query)

def parse_and_fix_manchester(query: str) -> str:
    individuals = {
        "1_stars", "2_stars", "3_stars", "4_stars", "5_stars",
        "Sauna", "24h_front_desk", "Airport_Shuttle", "Free_Wifi",
        "Parking_Space", "Private_Bathroom", "RestaurantInHotel", "Swimming_Pool",
        "FastFood", "Fusion", "Michelin", "StreetFood", "Traditional"
    }

    pattern = r"(\w+)\s+(some|value)\s+([\w\s]+?)(?=\s*\)|$)"

    def replacement(match):
        prop = match.group(1)
        op = match.group(2)
        target = match.group(3).strip()

        if target in individuals:
            return f"{prop} value {target}"
        else:
            return f"{prop} some {target}"

    return re.sub(pattern, replacement, query)

# test

# query = "Hotel and (inCountry value Italy) and (nextTo some BodyOfWater) and (inCity value Rome)"
# x = resolve_manchester_fillers(query)
# print(resolve_manchester_fillers(query))

In [6]:
NS1 = "http://kai.cs.vu.nl/2024/situated-minor-project/hotel#"

def get_iri(identifier):
    # Strip any remaining parentheses from the start or end of the string
    identifier = identifier.strip("()")
    if identifier.startswith("http"):
        return IRI.create(identifier)
    return IRI(NS1, identifier)

def parse_manchester_query(query_str):
    # Split by 'and', but if no 'and' exists, parts will just be a list with one item
    parts = re.split(r'\s+and\s+', query_str)
    expressions = []

    for part in parts:
        # Clean the part: remove leading/trailing whitespace and outer parentheses
        part = part.strip().strip("()")

        if " some " in part:
            prop_str, filler_str = part.split(" some ")
            prop = OWLObjectProperty(get_iri(prop_str))
            filler = OWLClass(get_iri(filler_str))
            expressions.append(OWLObjectSomeValuesFrom(property=prop, filler=filler))

        elif " value " in part:
            prop_str, filler_str = part.split(" value ")
            prop = OWLObjectProperty(get_iri(prop_str))
            filler = OWLNamedIndividual(get_iri(filler_str))
            # Fixed: parameter name is 'value', not 'individual'
            expressions.append(OWLObjectHasValue(property=prop, individual=filler))

        else:
            expressions.append(OWLClass(get_iri(part)))

    # If there is only one expression, return it directly instead of an IntersectionOf
    if len(expressions) == 1:
        return expressions[0]

    return OWLObjectIntersectionOf(expressions)

# Test with a single expression
query_single = "(inCountry value http://www.wikidata.org/entity/Q38) and Hotel"
complex_ce = parse_manchester_query(query_single)
print(f"Single Query Output: {complex_ce}")

# Test with a full conjunction
query_full = "(nextTo some BodyOfWater)"
complex_ce_full = parse_manchester_query(query_full)
print(f"Full Query Output: {complex_ce_full}")

Single Query Output: OWLObjectIntersectionOf((OWLObjectHasValue(property=OWLObjectProperty(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'inCountry')), individual=OWLNamedIndividual(IRI('http://www.wikidata.org/entity/', 'Q38'))), OWLClass(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'Hotel'))))
Full Query Output: OWLObjectSomeValuesFrom(property=OWLObjectProperty(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'nextTo')),filler=OWLClass(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'BodyOfWater')))


# Create your own dialogues

Once you have created your ontology, use it as the foundation for designing dialogues. Study the examples in examples.txt to understand their structure and content. Then, create 10 dialogues of your own, ensuring a range of difficulty levels: 5 simple ones and 5 more challenging ones. These dialogues should illustrate how your ontology can support reasoning and should include references to the types of information modeled in your ontology.

In [7]:
# Create 10 dialoges based on the description

dialogue1: str = "Find me an accomodation that is in Portugal and near a landmark." #exists in ABox
dialogue2: str = "I want to go to a camping by a river which is rated 5_stars."
dialogue3: str = "What are arabic restaurants that serve fastfood."
dialogue4: str = "Find me all accomodations rated 5 stars."
dialogue5: str = "Hotels near the sea."
dialogue6: str = "I want a hotel with a sauna and a swimming pool, preferrably in Portugal near a train station." #for instance acco0
dialogue7: str = "Find me a hotel that is also a michelin restaurant in France. They should have free parking and a 24h front desk too."
dialogue8: str = "Can you find me a hostel in Loures with private bathrooms. It would be nice if it is within walking distance from a museum."
dialogue9: str = "I want a Chinese Restaurant in Turkey right by the sea with parking space."
dialogue10: str = "I want a hotel that has a restaurant with private bathrooms. Preferrably rated highly."
dialogues: list = [dialogue1, dialogue2, dialogue3, dialogue4, dialogue5, dialogue6, dialogue7, dialogue8, dialogue9, dialogue10]

# 3. Deploy a reasoning environment

Treated as the ABox in the OWL knowledge base. The idea is that instance queries
with complex class expressions should be used to retrieve different hotels, where
reasoning is crucial for many aspects. For example, a query "give me places that are
close to a coast" would also return places next to a beach if the query is evaluated
together with the ontology that states that a place next to the beach is next to a
coast.

In [8]:
namespace = "http://kai.cs.vu.nl/2024/situated-minor-project/hotel#"

# Only explicit cases
pellet_reasoner = SyncReasoner(
    ontology=owl_file_path,
    reasoner="Pellet"
    )

# acutally infers axioms (triples)
structural_reasoner = StructuralReasoner(
    ontology=owl_file_path,
    property_cache = True, negation_default = True, sub_properties = False
    )

# 4. Instruct the LLM to produce the query or components of the query (e.g., keywords) against the KG

In [9]:
instruction : str = """### Role
You are a specialized NLP engine. Your sole task is to translate natural language user requests into formal Manchester Description Logic (DL) queries.

### Syntax Rules
1. **Format:** 'Class and (property some Class)' OR 'Class and (property value Individual)'.
2. **Naming:** Use PascalCase for Classes (e.g., Camping_Site) and camelCase for properties (e.g., inCountry).
3. **Operators:** and, or, not, some, value.
4. **Logic Selection:**
   - Use `value` for specific names (Italy, Vegan, Traditional) or specific facilities.
   - Use `some` for general categories (Sea, Restaurant, BodyOfWater).
5. **Nesting:** If a user mentions a feature within a place (e.g., a hotel with a gym), nest it: `Hotel and (hasFacility value Gym)`.

### Vocabulary Reference
- **Properties:** hasFacility, location, userRating, nationality, restaurantType, inCity, diet, nextTo, inCountry.
- **Classes:** Restaurant, Hotel, Camping_Site, Hostel, Neighbourhood, Museum, City, TrainStation, PublicTransport, BodyOfWater, TouristAttraction, RestaurantType, UserRating, Country, Diet, Accommodation, River,
Gold River, Ocean, Lagoon, Drainage Basin, Bay, Gulf, Sea, Adjacent Sea, Mediterranean Sea,
Watercourse, Canal, Main Stream.
- **Facility Values:** Sauna, 24h_front_desk, Airport_Shuttle, Free_Wifi, Parking_Space, Private_Bathroom, RestaurantInHotel, Swimming_Pool.
- **RestaurantType Values:** FastFood, Fusion, Michelin, StreetFood, Traditional.
- **UserRating Values:** 1_stars, 2_stars, 3_stars, 4_stars, 5_stars.

### Translation Logic
- "Near" or "By" -> `nextTo`
- "In [Country]" -> `inCountry value [Country]`
- "Serving [Diet] food" -> `diet value [Diet]`
- "rating of atleast 4 stars" or "rated highly" -> `userRating some HighRating`

### Output Requirement
- Output ONLY the query string.
- No explanations, no introductory text.
- No full stop.

### Examples
- User: "Hotel in Paris with a pool and a sauna."
- Query: Hotel and (inCity value Paris) and (hasFacility value Swimming_Pool) and (hasFacility value Sauna)

- User: "A restaurant next to a museum serving vegan food."
- Query: Restaurant and (nextTo some Museum) and (diet value Vegan)

- User: "Camping near a gold river in France close by some public transport."
- Query: "Camping_Site" and (nextTo some Gold River) and (inCountry value France) and (nextTo some PublicTransport)

- User: "Hotel with a high rating."
- Query: Hotel and (userRating some HighRating)

- User: "Hostel in Germany next to a river."
- Query: Hostel and (inCountry value Germany) and (nextTo some River)

- User: "I want to find chinese restaurants that serve vegan food."
- Query: Restautant and (nationality value Chinese) and (diet value Vegan)

- User: "Hotel rated 5 stars."
- Query: Hotel and (userRating value 5_stars)
"""
# To add ?
"""
- "Without [Feature]" -> `and not (property value/some [Feature])`
"""

'\n- "Without [Feature]" -> `and not (property value/some [Feature])`\n'

In [ ]:
user_input : str = "Find me all hotels next to a river in Lisbon with a sauna."

In [11]:
#Step 2: Write a function that takes the model, instruction and one user question as input, runs the LLM and outputs its response
def question_to_query(instruction: str, question: str, model="llama3.2") -> str:
    '''
    This function is meant to use the instruction defined above to run the LLM in order to convert one user input
    question into a query for the ontology reasoner.
    Parameters: instruction (string), question (string), model version (string)
    Returns: LLM response (string)
    '''

    # get initial response to generate manchester syntax query
    response : str = ollama.chat(
        model=model,
        messages=[
            {'role': 'system', 'content': instruction},
            {'role': 'user', 'content': question},
        ],
        options={
            'temperature': 0
        }
    )
    manchester_syntax_query = response['message']['content']
    return manchester_syntax_query


In [ ]:
question_to_query(instruction, user_input)

NameError: name 'instruction' is not defined

In [ ]:
def correct_query(query: str, model="llama3.2") -> str:
    #     - **Properties:** hasFacility, location, userRating, nationality, restaurantType, inCity, diet, nextTo, inCountry.
    validation_prompt = f"""
    ### Task
    Review the following Manchester Syntax query. Fix it ONLY if it violates the some/value rules.

    ### Rules
    1. Use 'value' when the object is a specific individual (e.g., 5_stars, Paris, Vegan, Swimming_Pool,).
    2. Use 'some' when the object is a general Class (e.g., River, Museum, HighRating, BodyOfWater).

    ### Reference
    - **Individuals (Use 'value'):**
    UserRating: 1_stars, 2_stars, 3_stars, 4_stars, 5_stars.
    Facility Values: Sauna, 24h_front_desk, Airport_Shuttle, Free_Wifi, Parking_Space, Private_Bathroom, RestaurantInHotel, Swimming_Pool.
    RestaurantType: FastFood, Fusion, Michelin, StreetFood, Traditional.
    - **Classes (Use 'some'):** Restaurant, Hotel, Camping_Site, Hostel, Neighbourhood, Museum, City, TrainStation, PublicTransport, BodyOfWater, TouristAttraction, RestaurantType, UserRating, Country, Diet, Accommodation, River,
    Gold River, Ocean, Lagoon, Drainage Basin, Bay, Gulf, Sea, Adjacent Sea, Mediterranean Sea,
    Watercourse, Canal, Main Stream.

    ### Input Query
    {query}

    ### Output Requirement
    - Output ONLY the corrected query string.
    - If it is already correct, output the original string.
    - No explanations or punctuation.

    ### Example
    - Input query: Hotel and (nextTo value Sea)
    - Corrected output query: Hotel and (nextTo some Sea)

    - Input query: Restaurant and (userRating some 5_stars)
    - Corrected output query: Restaurant and (userRating value 5_stars)
    """

    response = ollama.chat(
        model=model,
        messages=[
            {'role': 'user', 'content': validation_prompt}
        ],
        options={
            'temperature': 0
        }
    )

    return response['message']['content']



In [12]:
parse_and_fix_manchester("Camping_Site and (nextTo value River) and (userRating some 5_stars)")

'Camping_Site and (nextTo some River) and (userRating value 5_stars)'

In [ ]:
#Step 3: Run the LLM for each example defined above

# Helper function
def find_between(s: str, start: str, end: str) -> str:
    return s.split(start)[1].split(end)[0]

queries=[]

for dialogue in dialogues:
    print("User question:", dialogue)
    print()
    result: str = question_to_query(instruction, dialogue)
    # Possibly only extract the relevant parts
    print("Extracted query:", result)
    queries.append(result)
    print()



User question: Find me an accomodation in Portugal which is near a landmark.

Extracted query: Accommodation and (near some Landmark) in Portugal

User question: I want to go to a camping which has free Wifi and near public transport

Extracted query: Camping_Site and (hasFacility value Free_Wifi) and (nextTo some PublicTransport)

User question: What are restaurantrs that serve vegan food.

Extracted query: Restaurant and (diet value Vegan)

User question: Find me all accomodations rated 5 stars.

Extracted query: Accommodation and (userRating some 5_stars)

User question: Restaurants next to a river.

Extracted query: Restaurant and (nextTo some BodyOfWater)

User question: I want a hotel with a sauna and a swimming pool, preferrably in Spain near the sea.

Extracted query: Hotel and (hasFacility value Sauna) and (hasFacility value Swimming_Pool) and (inCountry value Spain) and (nextTo some Sea).

User question: Find me a hotel that is also a michelin restaurant in France. They shoul

# 5. Use an LLM to summarize some result into a natural language response to the user.

In [23]:
#Step 1: Extract knowledge from query with the reasoner and return as list
def reason(query: str, namespace: str = "http://kai.cs.vu.nl/2024/situated-minor-project/hotel#") -> list:
    '''
    This function should convert a query into an OWL expression and use the reasoner
    to return the answers.
    Uses OWLAPY structural reasoner initialized earlier. Converts Manchester Syntax Query into OWL
    class expression.
    Converts generator object to list.
    '''
    # try:
    #     manchester_to_owl_expression(query, namespace)
    # except (IncompleteParseError) as e:
    #     print("Please provide a correctly formatted input query: " \
    #     f"{query} --- is not correctly formatted")

    query_resolved = resolve_manchester_fillers(query) #swap wdt instances or classes by their correct uri
    query_some_value_fixed = parse_and_fix_manchester(query_resolved)
    print(query_some_value_fixed)
    owl_class_expression = parse_manchester_query(query_some_value_fixed) # convert manch query to OWL class expression

    print("Query converted to OWL ce: ",owl_class_expression)

    instances = []

    if pellet_reasoner.is_satisfiable(owl_class_expression):
        instances = list(pellet_reasoner.instances(owl_class_expression))
        if instances:
            return [i.iri for i in instances]
    else:
        print("Class expression is not satisfiable.")

    return instances # return default empty results list

##### Test case: complex query

In [15]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(10)

!ollama pull llama3.2

In [ ]:
q = "Hotel and (nextTo some River) and (inCity value Lisbon) and (hasFacility value Sauna)"
_q = "Hotel and (hasFacility value Sauna) and (hasFacility value Swimming_Pool) and (inCountry value Portugal)"

query_corrected = resolve_manchester_fillers(_q) #swap wdt instances or classes by their correct uri
owl_class_expression = parse_manchester_query(query_corrected) # convert manch query to OWL class expression

if pellet_reasoner.is_satisfiable(owl_class_expression):
    print(list(pellet_reasoner.instances(owl_class_expression)))

[OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation105')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation796')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation820')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation421')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation450')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation170')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation785')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation498')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'accomodation199')), OWLNamedIndividual(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 

In [16]:
from owlapy.iri import IRI
from owlapy.vocab import OWLRDFVocabulary
from owlapy.owl_annotation import OWLAnnotationObject
from owlready2 import *
from owlapy import owl_individual

In [17]:
onto = get_ontology(owl_file_path).load() #from owlready2

In [18]:
#Step 2: Instruct & run the LLM for the new task: transform the extracted knowledge into a natural language response based
# on the original question

def get_label(iri):
    entity = default_world[iri]
    if entity:
        if hasattr(entity, "label") and entity.label:
            return entity.label[0]
        return entity.name
    return iri

def knowledge_to_response(question: str, query: str, knowledge: list[str], model="llama3.2"):
    '''
    This function is meant to write an instruction based on an item of extracted knowledge and the original user
    question, and run the LLM to summarize a response.
    '''
    knowledge_enriched = [iri._remainder for iri in knowledge]
    print(knowledge_enriched)
    prompt: str = f"""You are answering a user question using ONLY the retrieved knowledge below.

    Important:
    - Each bullet is a MATCHING INDIVIDUAL from the knowledge graph (i.e., those are the entities that satisfy the query).
    - You may only use the facts shown in the bullets (labels / properties / etc.).
    - The "retrieved matching individuals" is ALREADY the answer to the "question" - you do not need to speculate about the truth of the answer, just explain the answer already given to you.
    - List the criteria provided to you in the query in bullet points before providing the retrieved knowledge. These should be nicely formatted in natural language.

    Query:
    {query}

    Retrieved matching individuals:
    {knowledge_enriched}

    Rules:
    - If the list is empty: say no matches found. This ONLY applies if under the matching individiuals it says "(empty)"
    - If bullets lack details needed to answer: say what detail is missing.
    - Otherwise answer in 1-3 sentences, concise and direct.
    """
    response : str = ollama.chat(
      model=model,
      messages=[
          {'role': 'system', 'content': prompt},
          {'role': 'user', 'content': question},
      ],
      options={
          'temperature': 0
      }
    )
    knowledge_answer = response['message']['content']
    return knowledge_answer


In [19]:
#Step 3: Combine everything: generate queries from the dialogues, extract knowledge from queries with the reasoner and
# generate summary responses

def run_pipeline(questions:list[str]) -> list[str]:

    for ind, question in enumerate(questions):
        print(30*"=")
        query = question_to_query(instruction, question)
        print("Ollama generated query:", query)
        knowl = reason(query)
        print("IRIs extracted from knowledge graph:", knowl)
        result = knowledge_to_response(
            question,
            query,
            knowl
        )
        print(f"Question {ind+1}: \n'{question}'.\nKnowledge Enriched Answer:\n{result}")
        print(30*"=")
        print()


In [24]:
ke_responses = run_pipeline(dialogues)

Ollama generated query: Accommodation and (inCountry value Portugal) and (nextTo some Landmark)
Accommodation and (inCountry value http://www.wikidata.org/entity/Q45) and (nextTo some Landmark)
Query converted to OWL ce:  OWLObjectIntersectionOf((OWLClass(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'Accommodation')), OWLObjectHasValue(property=OWLObjectProperty(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'inCountry')), individual=OWLNamedIndividual(IRI('http://www.wikidata.org/entity/', 'Q45'))), OWLObjectSomeValuesFrom(property=OWLObjectProperty(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'nextTo')),filler=OWLClass(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'Landmark')))))
IRIs extracted from knowledge graph: []
[]
Question 1: 
'Find me an accomodation that is in Portugal and near a landmark.'.
Knowledge Enriched Answer:
Here are the criteria you provided:

* Accommodation
* InCountry value Portugal
* NextTo some

KeyboardInterrupt: 

# 6. Evaluate your LLM

In [ ]:
# TODO: your code to implement and demonstrate evaluation metrics
# Suggestions: comparison of generated queries with the queries manually created in examples.txt, Intersection Over Union,
# but you can be creative here